# 13 - Node Embeddings ו-Link Prediction

מחברת זו לומדת ייצוג וקטורי בן 64 ממדים (embedding) עבור כל תחנה בגרף סמיכות הנסיעות (trip-adjacency) של התחבורה הציבורית בישראל, ולאחר מכן עושה שימוש בווקטורים הללו לשתי משימות: **link prediction** (האם ניתן לחזות אילו זוגות תחנות מחוברים בקטע שירות?) ו**סיווג תחנות קריטיות** (האם ה-embeddings מקודדים קריטיות מבנית?). כמו כן היא מציעה חיבורים חדשים ובוחנת האם הוספתם אכן הופכת את הרשת לעמידה יותר.

**עיקרה של מחברת זו הוא תיקון מתודולוגי.** הגרסה המוקדמת של pipeline זה אימנה Node2Vec על הגרף ה*מלא* ולאחר מכן העריכה link prediction על קשתות שהוחזקו בצד לצורך בדיקה - אלא שקשתות אלו עדיין היו נוכחות בזמן לימוד ה-embeddings. המודל למעשה כבר ראה את התשובות. ערך ה-AUC המתקבל, **0.9928** (לעומת 0.63-0.83 עבור מדדים קלאסיים), הוא ארטיפקט של דליפה (leakage) ולא תוצאה. כאן אנו מפצלים את הקשתות **תחילה**, מאמנים את ה-embeddings **רק על גרף האימון**, ומדווחים את המספר הכן לצד המספר הדולף.

**שאלות המחקר**
1. עד כמה ניתן לחזות את קטעי השירות החסרים בגרף התחבורה, באמצעות מדדים מבניים מקומיים לעומת embeddings נלמדים?
2. איזה חלק מן היתרון של ה-embeddings שדווח בעבר שורד הערכה נטולת דליפה?
3. האם קישורים חדשים המוצעים על ידי ה-embeddings משפרים באופן מדיד את חוסן הרשת?
4. האם node embeddings בלבד יכולים לזהות תחנות קריטיות מבחינה מבנית?

**קלט**
- `outputs/nb/02_graph_construction/` - גרף סמיכות הנסיעות הבלתי מכוון (pickle, או `nodes.csv` + `edges.csv`).
- חלופה במקרה ששלב 02 לא הורץ: פיד ה-GTFS הגולמי בתיקייה `israel-public-transportation/` יחד עם `stop_times.txt` (816 MB, מורד לפי דרישה). המחברת בונה אז את הגרף מחדש בעצמה, ולכן היא עצמאית לחלוטין.

**פלט** (הכול תחת `outputs/nb/13_embeddings_link_prediction/`)
- `tables/link_prediction_results.csv` - AUC / average precision לכל שיטה, מתויג `corrected` או `leaked`.
- `tables/link_prediction_hard_negatives.csv` - אותן שיטות מול זוגות שליליים קשים יותר (במרחק 2 צעדים).
- `tables/top_k_suggested_links.csv` - 20 החיבורים החדשים המוצעים המדורגים גלובלית.
- `tables/resilience_improvement.csv` ו-`tables/resilience_improvement.json` - חוסן לפני ואחרי (תוצאה אפסית).
- `tables/critical_classifier_results.csv` - ציוני המסווג (תוצאה שלילית).
- `tables/station_similarity.csv` - התחנות הקרובות ביותר במרחב ה-embedding.
- `embeddings_full_graph.npz`, `embeddings_train_graph.npz` - שתי מטריצות ה-embedding.
- `figures/*.png` - השוואת שיטות, מפת הקישורים המוצעים, חוסן, מטריצות בלבול, ותצוגות t-SNE ו-PCA.

## אתחול סביבת העבודה

מאתר את המאגר (או משכפל אותו בהרצה על Google Colab), מוודא שספריית העבודה היא שורש המאגר, ויוצר את עץ תיקיות הפלט של המחברת. ניתן להרצה חוזרת בבטחה.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## ספריות

נדרשת ערכת הכלים המדעית הרגילה, בתוספת `gensim` (skip-gram Word2Vec, המנגנון שהופך מסלולים אקראיים ל-embeddings) ובאופן אופציונלי החבילה `node2vec`. `node2vec` מוגדרת כאופציונלית במכוון: היא עטיפה דקה סביב `gensim`, היא נשברת מדי פעם מול גרסאות חדשות של `scipy`/`gensim`, ועם `p = q = 1` (ההגדרה שבה השתמש ה-pipeline המקורי) המסלולים מסדר שני שלה מתכנסים בדיוק למסלולים אקראיים משוקללים רגילים מסדר ראשון, שאותם אנו יכולים לייצר בעצמנו בכמה שורות. אם החבילה אינה זמינה, אנו נסוגים למחולל המסלולים הפנימי והמודל נותר זהה.

In [ ]:
_ensure("numpy", "pandas", "networkx", "scikit-learn", "matplotlib", "seaborn", "gensim")
try:
    _ensure("node2vec")
    HAS_NODE2VEC = True
except Exception as exc:
    print("node2vec could not be installed; the inline walk generator will be used:", exc)
    HAS_NODE2VEC = False

import csv, json, math, pickle, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             classification_report, confusion_matrix)

print("networkx", nx.__version__)
print("node2vec package available:", HAS_NODE2VEC)

## עיבוד תוויות בעברית

שמות התחנות בפיד ה-GTFS הם בעברית. Matplotlib אינה מממשת את אלגוריתם ה-Unicode הדו-כיווני (bidirectional), ולכן טקסט מימין לשמאל מצויר הפוך. התיקון שלהלן מפעיל את אלגוריתם ה-bidi על כל אובייקט טקסט פעם אחת, לפני ציור כל תרשים. כל הטקסט המילולי, הקוד וכותרות התרשימים במחברת הם באנגלית; רק ערכי הנתונים הם בעברית.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.0)
install_hebrew()  # seaborn resets rcParams, so re-apply the font settings

## קבועים ותקציב זמן ריצה

כל פרמטר יקר חושף כאן כדי שניתן יהיה להוזיל את המחברת בלי לערוך את קוד הניתוח. עלות משוערת בזמן שעון על מעבד של מחשב נייד עבור הגרף המלא (כ-30,000 צמתים וכ-52,000 קשתות):

| שלב | קבוע | עלות |
|---|---|---|
| Node2Vec, גרף האימון | `EMBEDDING_DIM`, `WALK_LENGTH`, `NUM_WALKS`, `EPOCHS` | 3-10 דקות |
| Node2Vec, הגרף המלא (נדרש להדגמת הדליפה ולכל השימושים התיאוריים) | זהה | 3-10 דקות |
| ציוני CCPA (מסלולים קצרים ביותר לכל זוג) | `CCPA_MAX_PAIRS` | 1-4 דקות |
| betweenness מקורב עבור תוויות הקריטיות | `BETWEENNESS_SAMPLES` | 1-3 דקות |
| שכנים קרובים גלובליים לפי cosine על כל ה-embeddings | `SIM_TOP_M` | 1-2 דקות |
| t-SNE עבור תרשימי מרחב ה-embedding | `TSNE_SAMPLE` | 1-3 דקות |

סך הכול כ-15-35 דקות. הגדרת `RUN_LEAKY_REPLICATION = False` מדלגת על אחת משתי הרצות ה-Node2Vec, אך אז לא ניתן לשחזר את ההשוואה המתודולוגית המרכזית של מחברת זו, ולכן היא נותרת פעילה.

In [ ]:
SEED = 42

# --- Node2Vec / Word2Vec hyper-parameters (same values as the original pipeline) ---
EMBEDDING_DIM = 64      # embedding dimensionality
WALK_LENGTH = 20        # nodes visited per random walk
NUM_WALKS = 5           # walks started from every node
WINDOW = 5              # skip-gram context window
EPOCHS = 3              # Word2Vec passes over the walk corpus
N2V_WORKERS = 1         # 1 keeps the run bit-for-bit reproducible; >1 is faster but not
EMBED_BACKEND = "auto"  # "auto" | "node2vec" | "inline"

# --- Link-prediction protocol ---
TEST_FRAC = 0.10        # share of edges held out as positive test examples
NEG_RATIO = 1.0         # negatives per positive
MAX_TRAIN_PAIRS = 20000 # cap on pairs used to fit the link-prediction classifier
CCPA_ALPHA = 0.8        # alpha of the CCPA score (see the CCPA section)
CCPA_MAX_PAIRS = 2000   # CCPA needs a shortest path per pair, so it is subsampled
RUN_LEAKY_REPLICATION = True

# --- Suggested links / resilience ---
TOP_K_SUGGEST = 20
SIM_TOP_M = 25          # neighbours kept per node when searching for the global top-K
REMOVAL_K_GRID = [50, 200, 500, 1000, 2000]

# --- Criticality labels and visualisation ---
BETWEENNESS_SAMPLES = 300  # pivot nodes for approximate betweenness (exact is infeasible)
BETWEENNESS_QUANTILE = 0.90
TSNE_SAMPLE = 6000

STAGE = OUT / "13_embeddings_link_prediction"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
print("Stage folder:", STAGE)

## טעינת הגרף

מושא המחקר הוא **גרף סמיכות הנסיעות (trip-adjacency graph)**: צומת הוא תחנה המופיעה בלפחות נסיעה אחת, וקשת בלתי מכוונת בין שתי תחנות מציינת שקיימת נסיעה המשרתת אותן ברצף. משקל הקשת הוא מספר הנסיעות העושות שימוש בקטע זה.

תחילה אנו מנסים לעשות שימוש חוזר בתוצרי המחברת `02_graph_construction` (גרף מסוג pickle, או `nodes.csv` + `edges.csv`). אם אלו אינם קיימים, התא הבא בונה את הגרף מחדש ישירות מפיד ה-GTFS הגולמי, כך שניתן להריץ מחברת זו באופן עצמאי.

In [ ]:
PREV = OUT / "02_graph_construction"

def _to_float(x):
    """Best-effort float conversion; returns None for blanks and NaN."""
    try:
        v = float(x)
    except (TypeError, ValueError):
        return None
    return v if v == v else None

def _graph_from_csv(nodes_csv, edges_csv):
    """Rebuild the undirected weighted graph from stage-02 CSV exports."""
    edges = pd.read_csv(edges_csv, encoding="utf-8-sig")
    cols = list(edges.columns)
    src = "from_stop" if "from_stop" in cols else cols[0]
    dst = "to_stop" if "to_stop" in cols else cols[1]
    wcol = next((c for c in ("trip_frequency", "weight", "count") if c in cols), None)
    arr = edges[[src, dst]].astype(str).values
    wts = edges[wcol].astype(float).values if wcol else np.ones(len(edges))
    H = nx.Graph()
    for (u, v), w in zip(arr, wts):
        if u == v:
            continue
        if H.has_edge(u, v):
            H[u][v]["weight"] += float(w)
        else:
            H.add_edge(u, v, weight=float(w))
    if nodes_csv is not None and Path(nodes_csv).exists():
        nodes = pd.read_csv(nodes_csv, dtype=str, encoding="utf-8-sig", keep_default_na=False)
        for r in nodes.to_dict("records"):
            nid = str(r.get("stop_id", ""))
            if nid in H:
                H.nodes[nid].update({
                    "stop_name": r.get("stop_name", ""),
                    "lat": _to_float(r.get("lat", r.get("stop_lat"))),
                    "lon": _to_float(r.get("lon", r.get("stop_lon"))),
                    "region": r.get("region", ""),
                    "metro": r.get("metro", ""),
                })
    return H

def load_stage02_graph():
    """Return the undirected stage-02 graph, or None when the stage has not been run."""
    if not PREV.exists():
        return None
    for p in sorted(PREV.glob("**/*.pkl")):
        name = p.name.lower()
        if "directed" in name and "undirected" not in name:
            continue
        try:
            with open(p, "rb") as f:
                obj = pickle.load(f)
        except Exception as exc:
            print("Could not read", p, "-", exc)
            continue
        if isinstance(obj, nx.Graph) and not obj.is_directed():
            print("Loaded stage-02 graph from", p)
            return obj
    edges_csv = next(iter(sorted(PREV.glob("**/edges.csv"))), None)
    nodes_csv = next(iter(sorted(PREV.glob("**/nodes.csv"))), None)
    if edges_csv is not None:
        print("Rebuilding graph from stage-02 CSVs:", edges_csv)
        return _graph_from_csv(nodes_csv, edges_csv)
    return None

G = load_stage02_graph()
if G is None:
    print("No stage-02 artifacts under", PREV)
    print("Falling back to building the graph from the raw GTFS feed in the next cell.")
    print("Run notebook 02_graph_construction first to skip the 816 MB download.")
else:
    print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

### חלופה: בניית הגרף מחדש מפיד ה-GTFS הגולמי

תא זה מבצע עבודה רק אם התא הקודם לא מצא דבר. הקובץ `stop_times.txt` שוקל 816 MB ולכן אינו מנוהל ב-git; הוא מורד לפי דרישה. הקובץ עוקב אחר מוסכמת GTFS ולפיה הוא ממוין לפי `trip_id` ולאחר מכן לפי `stop_sequence`, ולכן שתי שורות עוקבות של אותה נסיעה מגדירות קטע מכוון אחד. אנו קוראים אותו כזרם, שורה אחר שורה (בלי לטעון אותו לזיכרון), סופרים כמה נסיעות עושות שימוש בכל קטע, ומכווצים את הספירות המכוונות לגרף בלתי מכוון וממושקל. קואורדינטות התחנות, שמותיהן ותווית אזור גסה מגיעים מ-`stops.txt`. צפו ל-3-6 דקות בתוספת זמן ההורדה.

In [ ]:
if G is None:
    # stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
    _ensure("gdown")
    import gdown
    STOP_TIMES = DATA / "stop_times.txt"
    if not STOP_TIMES.exists():
        gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                       output=str(STOP_TIMES), quiet=False)
    print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

    csv.field_size_limit(10_000_000)

    def assign_region(lat, lon):
        """Coarse geographic region, same cut-offs as the data-preparation stage."""
        if 31.70 <= lat <= 31.90 and 34.95 <= lon <= 35.30:
            return "Jerusalem"
        if lat > 32.50:
            return "North"
        if lat >= 31.55:
            return "Center"
        return "South"

    stops = pd.read_csv(DATA / "stops.txt", dtype=str, keep_default_na=False,
                        encoding="utf-8-sig")
    stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
    stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
    stops = stops.dropna(subset=["stop_lat", "stop_lon"])
    attr = {}
    for r in stops.to_dict("records"):
        lat, lon = float(r["stop_lat"]), float(r["stop_lon"])
        attr[str(r["stop_id"])] = {"stop_name": r.get("stop_name", ""), "lat": lat, "lon": lon,
                                   "region": assign_region(lat, lon), "metro": ""}

    t0 = time.time()
    edge_count = defaultdict(int)
    with open(STOP_TIMES, encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti, si = header.index("trip_id"), header.index("stop_id")
        prev_trip, prev_stop = None, None
        for i, row in enumerate(reader, 1):
            trip, stop = row[ti], row[si]
            if trip == prev_trip and prev_stop is not None and prev_stop != stop:
                edge_count[(prev_stop, stop)] += 1
            prev_trip, prev_stop = trip, stop
            if i % 4_000_000 == 0:
                print(f"  {i:,} rows read, {len(edge_count):,} directed segments so far")

    G = nx.Graph()
    for (u, v), c in edge_count.items():
        if G.has_edge(u, v):
            G[u][v]["weight"] += c
        else:
            G.add_edge(u, v, weight=float(c))
    blank = {"stop_name": "", "lat": None, "lon": None, "region": "", "metro": ""}
    for n in G.nodes():
        G.nodes[n].update(attr.get(n, blank))
    print(f"Built graph in {time.time() - t0:.0f}s: "
          f"{G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
else:
    print("Stage-02 graph already loaded - nothing to rebuild.")

## צמצום לרכיב הקשירות הגדול ביותר

link prediction ו-embeddings מבוססי מסלולים אקראיים הם בעלי משמעות רק בתוך אזור קשיר: מסלול אינו יכול לצאת מן הרכיב שלו, וציונים המבוססים על מסלולים קצרים ביותר אינם מוגדרים בין רכיבים שונים. לפיכך אנו עובדים על רכיב הקשירות הגדול ביותר (LCC), בדיוק כפי שעשו הסקריפטים המקוריים, וממירים את כל מזהי הצמתים למחרוזות כך שמפתחות הגרף, מפתחות ה-embedding ומזהי ה-CSV לעולם לא ייסתרו זה את זה.

In [ ]:
G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()

print(f"Whole graph : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges, "
      f"{len(components):,} components")
print(f"LCC         : {Gc.number_of_nodes():,} nodes "
      f"({Gc.number_of_nodes() / G.number_of_nodes() * 100:.2f}% of nodes), "
      f"{Gc.number_of_edges():,} edges")
print(f"LCC density : {nx.density(Gc):.6f}   mean degree: "
      f"{2 * Gc.number_of_edges() / Gc.number_of_nodes():.2f}")

## הבאג שאנו מתקנים: דליפה בין קבוצת האימון לקבוצת המבחן

ה-pipeline המקורי פעל בסדר הבא:

1. `08_node2vec.py` אימן Node2Vec **על הגרף המלא** וכתב את `embeddings_df.csv`.
2. `09_link_prediction.py` הסיר לאחר מכן 10% מהקשתות כדי לשמש כדוגמאות חיוביות למבחן, חישב את המדדים הקלאסיים על גרף האימון שנותר, וחישב את השיטה מבוססת ה-embedding עם ה-embeddings **משלב 1**.

שלב 2 הוא נקודת הכשל. הקשתות שהוחזקו בצד היו נוכחות בגרף בעת לימוד ה-embeddings, ולכן מסלולים אקראיים חצו את אותן קשתות ממש אלפי פעמים, ופונקציית המטרה של skip-gram קירבה במפורש את קצותיהן במרחב הווקטורי. לשאול את המודל בדיעבד האם זוגות אלו מחוברים פירושו לבקש ממנו לשלוף את נתוני האימון של עצמו. ערך ה-**AUC 0.9928** שדווח מודד שינון, לא חיזוי. המדדים הקלאסיים לא הושפעו (הם חושבו על גרף האימון), וזו הסיבה שההשוואה נראתה כה חד-צדדית: 0.9928 מול 0.63-0.83.

דליפה שנייה, קטנה יותר, שכנה בתוך אותה פונקציה עצמה: הרגרסיה הלוגיסטית שמעל ה-embeddings הותאמה על 70% מ*זוגות המבחן עצמם* והוערכה על 30% הנותרים, כך שגם למסווג הייתה גישה תוך-התפלגותית לקבוצת ההערכה.

**הפרוטוקול המתוקן שבו נשתמש מכאן ואילך:**

1. לפצל את קשתות ה-LCC ל-90% אימון / 10% חיוביים שמורים, ולבנות את `G_train` ללא הקשתות השמורות.
2. לדגום זוגות שליליים (שאינם קשת), זרים לכל זוג חיובי.
3. לאמן Node2Vec **על `G_train` בלבד**. הקשתות השמורות סמויות מעיניו.
4. להתאים את מסווג ה-link prediction על זוגות שנשאבו מקשתות ה*אימון* בתוספת הזוגות השליליים שלהן.
5. להעריך פעם אחת בלבד, על החיוביים השמורים ועל השליליים שלהם.

בהמשך אנו עדיין משחזרים את המערך הדולף, זה לצד זה, כדי שניתן יהיה לראות ישירות את גודל הארטיפקט.

In [ ]:
rng = random.Random(SEED)

edges = [(str(u), str(v)) for u, v in Gc.edges()]
rng.shuffle(edges)
n_test = int(len(edges) * TEST_FRAC)
test_pos = edges[:n_test]
train_pos_pool = edges[n_test:]

G_train = Gc.copy()
G_train.remove_edges_from(test_pos)

nodes_list = list(Gc.nodes())
existing = {frozenset(e) for e in Gc.edges()}   # every true edge, incl. the held-out ones
used_pairs = set()                              # pairs already handed out as negatives

def sample_negatives(k, rng, attempts_factor=50):
    """Random non-adjacent node pairs. The graph has density ~0.0001, so a random
    pair is almost never an edge; rejection sampling is far cheaper than enumerating
    the O(n^2) non-edges."""
    out, attempts = [], 0
    while len(out) < k and attempts < k * attempts_factor:
        attempts += 1
        u, v = rng.choice(nodes_list), rng.choice(nodes_list)
        if u == v:
            continue
        key = frozenset((u, v))
        if key in existing or key in used_pairs:
            continue
        used_pairs.add(key)
        out.append((u, v))
    return out

def sample_hard_negatives(k, rng, attempts_factor=200):
    """Non-edges whose endpoints are two hops apart in G_train: pairs that share a
    neighbour but are not connected. Much harder than random pairs."""
    out, attempts = [], 0
    while len(out) < k and attempts < k * attempts_factor:
        attempts += 1
        u = rng.choice(nodes_list)
        nbrs = list(G_train.neighbors(u))
        if not nbrs:
            continue
        mid = rng.choice(nbrs)
        nbrs2 = list(G_train.neighbors(mid))
        if not nbrs2:
            continue
        w = rng.choice(nbrs2)
        key = frozenset((u, w))
        if u == w or key in existing or key in used_pairs:
            continue
        used_pairs.add(key)
        out.append((u, w))
    return out

test_neg = sample_negatives(int(n_test * NEG_RATIO), rng)
n_lr = min(len(train_pos_pool), MAX_TRAIN_PAIRS)
lr_pos = rng.sample(train_pos_pool, n_lr)
lr_neg = sample_negatives(n_lr, rng)
hard_neg = sample_hard_negatives(n_test, rng)

isolated = sum(1 for n in G_train.nodes() if G_train.degree(n) == 0)
print(f"Held-out positives : {len(test_pos):,}")
print(f"Random negatives   : {len(test_neg):,}")
print(f"Hard (2-hop) negs  : {len(hard_neg):,}")
print(f"Classifier fit pairs: {len(lr_pos):,} positive + {len(lr_neg):,} negative "
      f"(all from the training graph)")
print(f"G_train            : {G_train.number_of_nodes():,} nodes, "
      f"{G_train.number_of_edges():,} edges, {isolated:,} nodes left with degree 0")

## לימוד ה-embeddings

Node2Vec דוגם מסלולים אקראיים על הגרף ומזין אותם ל-skip-gram Word2Vec, תוך התייחסות למסלול כאל משפט ולצומת כאל מילה. צמתים המופיעים שוב ושוב באותם הקשרי מסלול מסתיימים קרובים זה לזה במרחב בן 64 הממדים.

ה-pipeline המקורי השתמש ב-`p = 1, q = 1`. עם ערכים אלו ההטיה מסדר שני של Node2Vec מתאפסת והמסלולים הם מסלולים אקראיים משוקללים רגילים מסדר ראשון, ולכן שני ה-backends שלהלן מהווים מודלים שקולים: החבילה `node2vec` אם ייבואה הצליח, ואחרת מחולל מסלולים פנימי בתוספת `gensim.Word2Vec`. שניהם עוקבים אחר משקלי הקשתות (תדירות הנסיעות), כך שצירים עמוסים נדגמים לעיתים קרובות יותר.

אנו מאמנים **שני** מודלים:
- `emb_train` - על `G_train` בלבד. זהו המודל המשמש להערכה הכנה של link prediction.
- `emb_full` - על ה-LCC כולו. לגיטימי לשימושים תיאוריים (ויזואליזציה, דמיון בין תחנות, סיווג קריטיות והצעת קישורים), ומשמש גם לשחזור ההערכה הדולפת.

כל הרצה נמשכת מספר דקות; `N2V_WORKERS = 1` שומר על יכולת שחזור במחיר המהירות.

In [ ]:
def weighted_random_walks(H, num_walks, walk_length, seed):
    """First-order weighted random walks - identical to Node2Vec with p = q = 1."""
    walk_rng = random.Random(seed)
    adj, wts = {}, {}
    for n in H.nodes():
        nbrs = list(H.neighbors(n))
        adj[n] = nbrs
        wts[n] = [float(H[n][m].get("weight", 1.0)) for m in nbrs]
    order = list(H.nodes())
    walks = []
    for _ in range(num_walks):
        walk_rng.shuffle(order)
        for start in order:
            walk = [start]
            while len(walk) < walk_length:
                cur = walk[-1]
                if not adj[cur]:
                    break
                walk.append(walk_rng.choices(adj[cur], weights=wts[cur], k=1)[0])
            walks.append([str(x) for x in walk])
    return walks

def train_embeddings(H, label, backend=EMBED_BACKEND):
    """Return {node_id (str): 64-d vector} for every node of H."""
    t0 = time.time()
    if backend in ("auto", "node2vec") and HAS_NODE2VEC:
        try:
            from node2vec import Node2Vec
            n2v = Node2Vec(H, dimensions=EMBEDDING_DIM, walk_length=WALK_LENGTH,
                           num_walks=NUM_WALKS, p=1, q=1, workers=N2V_WORKERS,
                           seed=SEED, quiet=True)
            model = n2v.fit(window=WINDOW, min_count=1, batch_words=4, epochs=EPOCHS)
            emb = {str(n): np.asarray(model.wv[str(n)], dtype=float)
                   for n in H.nodes() if str(n) in model.wv}
            print(f"  [{label}] node2vec package: {len(emb):,} vectors "
                  f"in {time.time() - t0:.0f}s")
            return emb
        except Exception as exc:
            if backend == "node2vec":
                raise
            print(f"  [{label}] node2vec package failed ({exc}); using inline walks")
    from gensim.models import Word2Vec
    walks = weighted_random_walks(H, NUM_WALKS, WALK_LENGTH, SEED)
    model = Word2Vec(walks, vector_size=EMBEDDING_DIM, window=WINDOW, min_count=1,
                     sg=1, workers=N2V_WORKERS, epochs=EPOCHS, seed=SEED)
    emb = {str(n): np.asarray(model.wv[str(n)], dtype=float)
           for n in H.nodes() if str(n) in model.wv}
    print(f"  [{label}] inline walks: {len(walks):,} walks, {len(emb):,} vectors "
          f"in {time.time() - t0:.0f}s")
    return emb

print("Training embeddings on the TRAINING graph (held-out edges removed) ...")
emb_train = train_embeddings(G_train, "train graph")

print("Training embeddings on the FULL LCC (descriptive uses + leakage demo) ...")
emb_full = train_embeddings(Gc, "full graph")

def save_embeddings(emb, path):
    ids = list(emb.keys())
    np.savez_compressed(path, ids=np.array(ids, dtype=object),
                        vectors=np.vstack([emb[i] for i in ids]))
    print("  saved", path.name, f"({len(ids):,} x {EMBEDDING_DIM})")

save_embeddings(emb_train, STAGE / "embeddings_train_graph.npz")
save_embeddings(emb_full, STAGE / "embeddings_full_graph.npz")

## ציוני link prediction קלאסיים

קווי הבסיס הם היוריסטיקות התקניות של חפיפת שכנויות, כולן מחושבות **על גרף האימון**:

- **Common Neighbors** - כמה שכנים משותפים לשתי התחנות.
- **Jaccard** - שכנים משותפים חלקי איחוד שתי השכנויות.
- **Adamic-Adar** - שכנים משותפים משוקללים ב-`1 / log(degree)`, כך שרכזות (hubs) נדירות נספרות במשקל רב יותר.
- **Resource Allocation** - שכנים משותפים משוקללים ב-`1 / degree`, גרסה חדה יותר של אותו רעיון.
- **Preferential Attachment** - מכפלת שתי הדרגות; מתעלם לחלוטין מחפיפה.

**תיקון תיוג (c).** הסקריפט המקורי חישב את שורת "Common Neighbors" שלו באמצעות `nx.common_neighbor_centrality`. פונקציה זו **אינה** מחזירה ספירת שכנים משותפים: עם ברירת המחדל `alpha = 0.8` היא מחזירה את ציון ה-CCPA `alpha * |common neighbours| + (1 - alpha) * N / d(u, v)`, המערבב בתוכו את *מרחק המסלול הקצר ביותר הגלובלי*. התיוג השגוי הסתיר את העובדה שקו בסיס זה עשה שימוש במידע שאף מדד מקומי אינו מחזיק בו. כאן אנו קוראים לה עם `alpha = 1`, שהיא ספירת השכנים המשותפים האמיתית, ומדווחים על CCPA בנפרד ותחת שמו האמיתי בסעיף הבא.

In [ ]:
def pair_scores(H, pairs):
    """Classical link-prediction scores for a list of node pairs, keyed by method."""
    out = {}
    out["Common Neighbors"] = {(u, v): s for u, v, s in
                               nx.common_neighbor_centrality(H, pairs, alpha=1)}
    out["Jaccard"] = {(u, v): s for u, v, s in nx.jaccard_coefficient(H, pairs)}
    out["Adamic-Adar"] = {(u, v): s for u, v, s in nx.adamic_adar_index(H, pairs)}
    out["Resource Allocation"] = {(u, v): s for u, v, s in
                                  nx.resource_allocation_index(H, pairs)}
    out["Preferential Attachment"] = {(u, v): s for u, v, s in
                                      nx.preferential_attachment(H, pairs)}
    return out

def lookup(d):
    """Undirected lookup into a score dictionary, defaulting to 0."""
    def _f(u, v):
        return d.get((u, v), d.get((v, u), 0.0))
    return _f

def evaluate(pos, neg, score_fn):
    """AUC-ROC and average precision for a scoring function over labelled pairs."""
    pairs = list(pos) + list(neg)
    labels = [1] * len(pos) + [0] * len(neg)
    scores = [float(score_fn(u, v)) for u, v in pairs]
    return (round(roc_auc_score(labels, scores), 4),
            round(average_precision_score(labels, scores), 4))

t0 = time.time()
classical = pair_scores(G_train, test_pos + test_neg)
print(f"Classical scores computed in {time.time() - t0:.0f}s")

results = []
for name, d in classical.items():
    auc, ap = evaluate(test_pos, test_neg, lookup(d))
    results.append({"method": name, "protocol": "corrected", "auc": auc,
                    "average_precision": ap})
    print(f"  {name:<26} AUC={auc:.4f}  AP={ap:.4f}")

## CCPA, מחושב ומתויג בכנות

CCPA (Common Neighbor and Centrality based Parameterized Algorithm, Ahmad et al. 2020) מדרג זוג לפי `alpha * |common neighbours| + (1 - alpha) * N / d(u, v)`. איבר המרחק הוא שגרם למדד זה להיראות חזק כל כך בהרצה המקורית: הזוגות השליליים שלנו הם שתי תחנות שנבחרו באקראי, המרוחקות בדרך כלל 20 צעדים ומעלה זו מזו, ואילו זוג חיובי שמור נמצא מעצם בנייתו במרחק 2 צעדים בגרף האימון ברוב המקרים. מדד שיכול לראות את `d(u, v)` מפריד בין שתי אוכלוסיות אלו כמעט ללא מאמץ - וזוהי תכונה של אופן דגימת הזוגות השליליים, ולא עדות לכך שהמדד חוזה קטעי שירות אמיתיים.

לא ניתן להשתמש ב-`nx.common_neighbor_centrality` ישירות בקנה מידה זה: היא מייצרת בזיכרון את מילון כל המסלולים הקצרים ביותר בין כל הזוגות (כ-30,000 x 30,000 רשומות) לפני שהיא מדרגת דבר. במקום זאת אנו מחשבים את המרחק רק עבור הזוגות שאנו באמת זקוקים להם, באמצעות BFS דו-כיווני, ודוגמים תת-קבוצה של `CCPA_MAX_PAIRS` זוגות כדי לשמור על זמן ריצה של דקה או שתיים.

In [ ]:
def ccpa_scores(H, pairs, alpha=CCPA_ALPHA):
    """CCPA score per pair, with per-pair bidirectional BFS instead of all-pairs SPL."""
    N = H.number_of_nodes()
    out = {}
    for u, v in pairs:
        cn = len(list(nx.common_neighbors(H, u, v)))
        try:
            d = nx.shortest_path_length(H, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            d = None
        out[(u, v)] = alpha * cn + ((1 - alpha) * N / d if d else 0.0)
    return out

ccpa_rng = random.Random(SEED)
k_side = max(1, CCPA_MAX_PAIRS // 2)
ccpa_pos = ccpa_rng.sample(test_pos, min(k_side, len(test_pos)))
ccpa_neg = ccpa_rng.sample(test_neg, min(k_side, len(test_neg)))

t0 = time.time()
ccpa = ccpa_scores(G_train, ccpa_pos + ccpa_neg)
auc, ap = evaluate(ccpa_pos, ccpa_neg, lookup(ccpa))
results.append({"method": f"CCPA (alpha={CCPA_ALPHA}, subsampled)", "protocol": "corrected",
                "auc": auc, "average_precision": ap})
print(f"CCPA on {len(ccpa_pos) + len(ccpa_neg):,} pairs in {time.time() - t0:.0f}s: "
      f"AUC={auc:.4f}  AP={ap:.4f}")
print("This is the row the original pipeline printed as 'Common Neighbors'.")

## link prediction מבוסס embedding, בביצוע נכון

זוג תחנות מומר לווקטור מאפיינים באמצעות **מכפלת Hadamard** של שני ה-embeddings (הכפלה איבר-איבר), האופרטור התקני ל-link prediction מבוסס Node2Vec. מעליו מותאמת רגרסיה לוגיסטית מאוזנת.

שתי הדליפות נסגרות כאן:
- ה-embeddings מגיעים מ-`emb_train`, שנלמד על `G_train`, ולכן הקשתות השמורות מעולם לא השפיעו עליהם;
- הרגרסיה הלוגיסטית מותאמת על זוגות שנשאבו מקשתות ה*אימון* בתוספת הזוגות השליליים שנדגמו עבורן, והזוגות השמורים נוגעים בתהליך פעם אחת בלבד, בשלב חישוב הציון.

In [ ]:
def hadamard_matrix(emb, pairs):
    """Stack Hadamard-product features; also returns the mask of usable pairs."""
    rows, keep = [], []
    for u, v in pairs:
        eu, ev = emb.get(str(u)), emb.get(str(v))
        if eu is None or ev is None:
            keep.append(False)
            continue
        rows.append(eu * ev)
        keep.append(True)
    return (np.vstack(rows) if rows else np.empty((0, EMBEDDING_DIM))), np.array(keep)

def embedding_link_prediction(emb, fit_pos, fit_neg, eval_pos, eval_neg):
    """Fit on (fit_pos, fit_neg), score (eval_pos, eval_neg). No overlap between them."""
    X_fit, keep_fit = hadamard_matrix(emb, list(fit_pos) + list(fit_neg))
    y_fit = np.array([1] * len(fit_pos) + [0] * len(fit_neg))[keep_fit]
    X_ev, keep_ev = hadamard_matrix(emb, list(eval_pos) + list(eval_neg))
    y_ev = np.array([1] * len(eval_pos) + [0] * len(eval_neg))[keep_ev]
    scaler = StandardScaler().fit(X_fit)
    clf = LogisticRegression(class_weight="balanced", max_iter=500, random_state=SEED)
    clf.fit(scaler.transform(X_fit), y_fit)
    prob = clf.predict_proba(scaler.transform(X_ev))[:, 1]
    return (round(roc_auc_score(y_ev, prob), 4),
            round(average_precision_score(y_ev, prob), 4),
            clf, scaler)

auc_corr, ap_corr, lp_clf, lp_scaler = embedding_link_prediction(
    emb_train, lr_pos, lr_neg, test_pos, test_neg)
results.append({"method": "Node2Vec + LR", "protocol": "corrected",
                "auc": auc_corr, "average_precision": ap_corr})
print(f"Node2Vec + LR, leakage-free: AUC={auc_corr:.4f}  AP={ap_corr:.4f}")

## שחזור ההערכה הדולפת

כדי להראות שההבדל נובע מן הפרוטוקול ולא משינוי אחר כלשהו, אנו מריצים כעת מחדש את המערך המקורי במדויק: embeddings מן הגרף ה**מלא**, ורגרסיה לוגיסטית המאומנת על 70% מזוגות המבחן ומוערכת על 30% הנותרים. כמו כן אנו מוסיפים קו בסיס קלאסי דולף במכוון - שכנים משותפים הנספרים על הגרף ה*מלא* - שאמור להיות כמעט מושלם מאותה סיבה, שכן כל קשת שמורה עדיין נוכחת בעת בחינת השכנויות של קצותיה. שתי השורות מתויגות `leaked` בטבלת התוצאות כך שאיש לא יטעה לראות בהן ממצאים.

In [ ]:
if RUN_LEAKY_REPLICATION:
    # (1) Original setup: full-graph embeddings, LR split inside the test pairs.
    X_all, keep_all = hadamard_matrix(emb_full, test_pos + test_neg)
    y_all = np.array([1] * len(test_pos) + [0] * len(test_neg))[keep_all]
    X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3,
                                              random_state=SEED, stratify=y_all)
    sc = StandardScaler().fit(X_tr)
    leak_clf = LogisticRegression(class_weight="balanced", max_iter=500, random_state=SEED)
    leak_clf.fit(sc.transform(X_tr), y_tr)
    p = leak_clf.predict_proba(sc.transform(X_te))[:, 1]
    auc_leak = round(roc_auc_score(y_te, p), 4)
    ap_leak = round(average_precision_score(y_te, p), 4)
    results.append({"method": "Node2Vec + LR", "protocol": "leaked",
                    "auc": auc_leak, "average_precision": ap_leak})
    print(f"Node2Vec + LR, leaked replication: AUC={auc_leak:.4f}  AP={ap_leak:.4f}")

    # (2) The same leak applied to a classical measure, for contrast.
    cn_full = {(u, v): s for u, v, s in
               nx.common_neighbor_centrality(Gc, test_pos + test_neg, alpha=1)}
    auc_cnl, ap_cnl = evaluate(test_pos, test_neg, lookup(cn_full))
    results.append({"method": "Common Neighbors (scored on full graph)",
                    "protocol": "leaked", "auc": auc_cnl, "average_precision": ap_cnl})
    print(f"Common Neighbors, leaked: AUC={auc_cnl:.4f}  AP={ap_cnl:.4f}")
    print()
    print(f"Leakage inflation for Node2Vec: {auc_leak:.4f} -> {auc_corr:.4f} "
          f"({auc_leak - auc_corr:+.4f} AUC).")
else:
    print("RUN_LEAKY_REPLICATION is False - skipping the demonstration.")

## טבלת התוצאות ותרשים ההשוואה

כל השיטות זו לצד זו. שורות המתויגות `leaked` פסולות מעצם בנייתן ומוצגות אך ורק לצורך ההשוואה המתודולוגית; הן מצוירות באדום בתרשים. הדירוג הכן הוא קבוצת השורות המתויגות `corrected`.

In [ ]:
res_df = pd.DataFrame(results)
res_df.to_csv(TABLES / "link_prediction_results.csv", index=False, encoding="utf-8-sig")
display(res_df.sort_values(["protocol", "auc"], ascending=[True, False])
        .reset_index(drop=True))

plot_df = res_df.copy()
plot_df["label"] = plot_df.apply(
    lambda r: r["method"] + ("  [LEAKED]" if r["protocol"] == "leaked" else ""), axis=1)
plot_df = plot_df.sort_values("auc")
colors = ["#dc2626" if p == "leaked" else "#2563eb" for p in plot_df["protocol"]]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(plot_df["label"], plot_df["auc"], color=colors)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="Random baseline (0.5)")
for i, v in enumerate(plot_df["auc"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0, 1.12)
ax.set_xlabel("AUC-ROC")
ax.set_title("Link prediction: leakage-free evaluation (blue) vs the original leaked setup (red)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES / "method_comparison_auc.png", dpi=150)
plt.show()

## מבחן קשה יותר: זוגות שליליים במרחק 2 צעדים

זוגות שליליים אקראיים הם קלים: שתי תחנות שנבחרו באופן אחיד באקראי נמצאות בדרך כלל בערים שונות. שאלה מציאותית יותר עבור מתכנן תחבורה היא *אילו מן הזוגות הסבירים אכן מחוברים* - זוגות שכבר חולקים שכן אך אין ביניהם קטע ישיר. אנו מחשבים מחדש את הציון של כל שיטה מול זוגות `hard_neg` אלו (הזוגות החיוביים נותרים ללא שינוי). אין צורך באימון מחדש: ה-embeddings והמסווג מעולם לא ראו אף אחד מזוגות אלו.

יש לצפות שכל ערכי ה-AUC יירדו. ירידה זו היא המדד הכן לשאלה איזה חלק מן הביצועים הקודמים נבע מדגימת הזוגות השליליים ולא מן המודל.

In [ ]:
hard_rows = []
hard_classical = pair_scores(G_train, test_pos + hard_neg)
for name, d in hard_classical.items():
    auc, ap = evaluate(test_pos, hard_neg, lookup(d))
    hard_rows.append({"method": name, "auc": auc, "average_precision": ap})

X_h, keep_h = hadamard_matrix(emb_train, test_pos + hard_neg)
y_h = np.array([1] * len(test_pos) + [0] * len(hard_neg))[keep_h]
prob_h = lp_clf.predict_proba(lp_scaler.transform(X_h))[:, 1]
hard_rows.append({"method": "Node2Vec + LR",
                  "auc": round(roc_auc_score(y_h, prob_h), 4),
                  "average_precision": round(average_precision_score(y_h, prob_h), 4)})

hard_df = pd.DataFrame(hard_rows)
easy = res_df[res_df["protocol"] == "corrected"].set_index("method")["auc"]
hard_df["auc_random_negatives"] = hard_df["method"].map(easy)
hard_df["delta"] = (hard_df["auc"] - hard_df["auc_random_negatives"]).round(4)
hard_df = hard_df.rename(columns={"auc": "auc_hard_negatives"})
hard_df.to_csv(TABLES / "link_prediction_hard_negatives.csv", index=False,
               encoding="utf-8-sig")
display(hard_df.sort_values("auc_hard_negatives", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, 5))
idx = np.arange(len(hard_df))
ax.barh(idx + 0.2, hard_df["auc_random_negatives"], height=0.4, color="#93c5fd",
        label="Random negatives")
ax.barh(idx - 0.2, hard_df["auc_hard_negatives"], height=0.4, color="#1d4ed8",
        label="2-hop (hard) negatives")
ax.set_yticks(idx)
ax.set_yticklabels(hard_df["method"])
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("AUC-ROC")
ax.set_title("Difficulty of the negative sample drives most of the reported AUC")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES / "hard_negatives_comparison.png", dpi=150)
plt.show()

## הצעת חיבורים חדשים (עם דירוג גלובלי תקין)

שאלת התכנון: אילו זוגות תחנות נראים דומים מבחינה מבנית (הם ממלאים אותו סוג של מיקום ברשת) אך אין ביניהם קטע ישיר? אלו הם קישורים חדשים מועמדים.

**תיקון (b).** הסקריפט המקורי דגם 500 תחנות אקראיות ודירג זוגות רק בתוך אותו מדגם, ואז הציג את התוצאה כ"עשרים המובילים" של הרשת. עם 30,000 תחנות, מדגם של 500 צמתים מכסה כ-0.03% מן הזוגות האפשריים, ולכן הרשימה שהודפסה הייתה שרירותית בעיקרה. כאן אנו מדרגים על פני **כל** קבוצת הצמתים: ה-embeddings מנורמלים ב-L2 (ואז דמיון cosine שווה למכפלה הסקלרית) וחיפוש שכן קרוב מדויק בכוח גס מחזיר עבור כל צומת את `SIM_TOP_M` השותפים הדומים לו ביותר; זוגות שכבר מהווים קשת מוסרים, והנותרים מדורגים גלובלית. מכיוון שזוג ה-top-20 הגלובלי חייב להופיע ברשימות ה-top-`SIM_TOP_M` של קצותיו עצמם, הליך זה משחזר במדויק את ה-top-20 הגלובלי, אלא אם לצומת כלשהו יש יותר מ-25 שותפים הדומים לו יותר מן הזוג ה-20 בטיבו בכל הרשת - דבר שאינו קורה ברמות דמיון אלו. עלות: חיפוש cosine מדויק אחד בגודל 30k x 30k, כדקה.

אנו מדווחים גם על מרחק המעגל הגדול (great-circle) של כל זוג מוצע, משום ש"קישור מוצע" בין שתי תחנות המרוחקות 100 ק"מ זו מזו אינו קטע אוטובוס שמישהו היה בונה.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return round(2 * R * math.asin(math.sqrt(a)), 2)

emb_ids = list(emb_full.keys())
M = np.vstack([emb_full[i] for i in emb_ids])
norms = np.linalg.norm(M, axis=1, keepdims=True)
norms[norms == 0] = 1.0
Mn = M / norms

t0 = time.time()
k_nn = min(SIM_TOP_M + 1, len(emb_ids))
nn = NearestNeighbors(n_neighbors=k_nn, metric="cosine", algorithm="brute").fit(Mn)
nn_dist, nn_idx = nn.kneighbors(Mn)
print(f"Exact cosine kNN over {len(emb_ids):,} nodes in {time.time() - t0:.0f}s")

cand = {}
for i, u in enumerate(emb_ids):
    for d, j in zip(nn_dist[i][1:], nn_idx[i][1:]):
        v = emb_ids[j]
        if u == v or Gc.has_edge(u, v):
            continue
        key = (u, v) if u < v else (v, u)
        cand[key] = max(cand.get(key, -1.0), 1.0 - float(d))

sug = pd.DataFrame([{"stop_a": a, "stop_b": b, "cosine_sim": round(s, 4)}
                    for (a, b), s in cand.items()])
sug = sug.sort_values("cosine_sim", ascending=False).head(TOP_K_SUGGEST).reset_index(drop=True)

def attr_of(n, key, default=""):
    return Gc.nodes[n].get(key, default) if n in Gc.nodes else default

for side in ("a", "b"):
    sug["name_" + side] = sug["stop_" + side].map(lambda n: attr_of(n, "stop_name"))
    sug["region_" + side] = sug["stop_" + side].map(lambda n: attr_of(n, "region"))
sug["distance_km"] = [haversine_km(attr_of(a, "lat", None), attr_of(a, "lon", None),
                                   attr_of(b, "lat", None), attr_of(b, "lon", None))
                      for a, b in zip(sug["stop_a"], sug["stop_b"])]
sug.to_csv(TABLES / "top_k_suggested_links.csv", index=False, encoding="utf-8-sig")
print(f"Candidate non-adjacent pairs considered: {len(cand):,}")
display(sug[["name_a", "name_b", "region_a", "region_b", "cosine_sim", "distance_km"]])

fig, ax = plt.subplots(figsize=(7.5, 10))
lats = [d.get("lat") for _, d in Gc.nodes(data=True) if d.get("lat") and d.get("lon")]
lons = [d.get("lon") for _, d in Gc.nodes(data=True) if d.get("lat") and d.get("lon")]
ax.scatter(lons, lats, s=1, color="#cbd5e1", alpha=0.4)
for r in sug.itertuples():
    la, lo = attr_of(r.stop_a, "lat", None), attr_of(r.stop_a, "lon", None)
    lb, lb2 = attr_of(r.stop_b, "lat", None), attr_of(r.stop_b, "lon", None)
    if None in (la, lo, lb, lb2):
        continue
    ax.plot([lo, lb2], [la, lb], "-", color="#dc2626", linewidth=1.5, alpha=0.8)
    ax.scatter([lo, lb2], [la, lb], s=25, color="#dc2626", zorder=5)
ax.set_title(f"Top {len(sug)} suggested links by embedding similarity (global ranking)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(FIGURES / "suggested_links_map.png", dpi=150)
plt.show()

## האם הקישורים המוצעים משפרים את החוסן? (תוצאה אפסית)

**תיקון (a).** הסקריפט המקורי כותרת סעיף זה "resilience improvement after adding suggested links", בעוד שהמספרים שלו עצמו היו `LCC share 0.9965 -> 0.9965`, כלומר **0.0%+**. התרשים עדיין צויר עם עמודת "after" ירוקה וכיתוב שנשא את המילה "improvement". זוהי תוצאה אפסית המוצגת כתוצאה חיובית.

אנו חוזרים על המדידה בכנות ומקשים על הסתרתה: במקום תקציב הסרה יחיד של 50 תחנות, אנו סורקים את `REMOVAL_K_GRID` ומדווחים את חלקו של ה-LCC (כשבר ממספר הצמתים המקורי) לפני ואחרי הוספת 20 הקשתות המוצעות, בכל תקציב. שתי סיבות לצפות להיעדר השפעה:

1. הסרת 50 תחנות מתוך כ-30,000 בקושי נוגעת בגרף כה דליל וכה גדול, ולכן קו הבסיס כבר עומד על כ-0.996 ואין מה לשפר.
2. עשרים קשתות נוספות הן 0.04% מ-52,000 הקשתות הקיימות, והן מחברות תחנות הדומות מבחינה מבנית - מה שבגרף זה נוטה להעיד שהן כבר נמצאות באותו אזור מקושר היטב, ולא משני צדדיו של חתך.

אם תופיע השפעה כלשהי בתקציבי ההסרה הגדולים יותר, זהו החלק המעניין; ואם לא, נאמר זאת במפורש.

In [ ]:
baseline_n = Gc.number_of_nodes()

def lcc_share(H, removed):
    """Largest-component size after removing `removed`, as a share of the original n."""
    T = H.copy()
    T.remove_nodes_from(removed)
    if T.number_of_nodes() == 0:
        return 0.0
    return max(len(c) for c in nx.connected_components(T)) / baseline_n

G_imp = Gc.copy()
for r in sug.itertuples():
    G_imp.add_edge(r.stop_a, r.stop_b, weight=0.002)

ranked = sorted(Gc.nodes(), key=lambda n: Gc.degree(n), reverse=True)
rows = []
for k in REMOVAL_K_GRID:
    before = lcc_share(Gc, ranked[:k])
    after = lcc_share(G_imp, ranked[:k])
    rows.append({"removal_k": k, "lcc_before": round(before, 4),
                 "lcc_after": round(after, 4),
                 "delta": round(after - before, 5),
                 "delta_pct": round((after - before) / before * 100, 3) if before else 0.0})
res_res = pd.DataFrame(rows)
res_res.to_csv(TABLES / "resilience_improvement.csv", index=False, encoding="utf-8-sig")
with open(TABLES / "resilience_improvement.json", "w", encoding="utf-8") as f:
    json.dump({"added_edges": int(len(sug)), "grid": rows}, f, ensure_ascii=False, indent=2)
display(res_res)

max_gain = res_res["delta"].abs().max()
print(f"Largest absolute change in LCC share across all removal budgets: {max_gain:.5f}")
print("Interpretation: " + ("no measurable effect - this is a null result."
                            if max_gain < 0.001 else
                            "a small but non-zero effect; see the table above."))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(res_res))
ax.bar(x - 0.2, res_res["lcc_before"], width=0.4, color="#94a3b8", label="Before added links")
ax.bar(x + 0.2, res_res["lcc_after"], width=0.4, color="#64748b", label="After added links")
for xi, (b, a) in enumerate(zip(res_res["lcc_before"], res_res["lcc_after"])):
    ax.text(xi, max(b, a) + 0.01, f"{a - b:+.4f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f"top-{k}" for k in res_res["removal_k"]])
ax.set_xlabel("Number of highest-degree stations removed")
ax.set_ylabel("LCC share of the original network")
ax.set_ylim(0, 1.1)
ax.set_title(f"Adding {len(sug)} suggested links changes resilience by ~0 (null result)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "resilience_before_after_links.png", dpi=150)
plt.show()

## תוויות קריטיות עבור ניסוי הסיווג

תחנה מתויגת כ**קריטית** אם היא נקודת חיתוך (articulation point - הסרתה מפצלת את הרכיב שלה) **או** אם מדד ה-betweenness centrality שלה נמצא ב-10% העליונים. זוהי ההגדרה שבה השתמש הסקריפט המקורי.

חישוב betweenness מדויק על 30,000 צמתים משמעו עץ מסלולים קצרים ביותר מכל צומת ואורך שעות, ולכן אנו משתמשים בקירוב ה-pivot התקני עם `BETWEENNESS_SAMPLES` צמתי מקור (כ-1-3 דקות). מכיוון שהסף הוא אחוזון של הערכים המקורבים, רעש דגימה קטן מזיז מספר תחנות מעבר לגבול; הדבר אינו משנה את המסקנות שלהלן, שאינן שוליות.

In [ ]:
t0 = time.time()
ap_set = set(nx.articulation_points(Gc))
print(f"Articulation points: {len(ap_set):,} ({time.time() - t0:.0f}s)")

t0 = time.time()
btw = nx.betweenness_centrality(Gc, k=min(BETWEENNESS_SAMPLES, Gc.number_of_nodes()),
                                seed=SEED)
btw_thr = float(np.quantile(list(btw.values()), BETWEENNESS_QUANTILE))
print(f"Approximate betweenness with {BETWEENNESS_SAMPLES} pivots "
      f"({time.time() - t0:.0f}s); p{int(BETWEENNESS_QUANTILE * 100)} = {btw_thr:.6f}")

critical = {n: int(n in ap_set or btw.get(n, 0.0) >= btw_thr) for n in Gc.nodes()}
n_crit = sum(critical.values())
print(f"Critical stations: {n_crit:,} / {len(critical):,} "
      f"({n_crit / len(critical) * 100:.1f}%)")

## האם ה-embeddings יכולים לזהות תחנות קריטיות? (תוצאה שלילית)

**תיקון (d).** ה-pipeline המקורי דיווח על ציוני F1 של 0.073 (רגרסיה לוגיסטית) ו-0.137 (random forest) עבור משימה זו ללא כל הערה. מספרים אלו קרובים לתוצאה שהיה משיג מסווג המנחש לפי שיעור הבסיס, ולכן ניסוי זה הוא **כישלון**, ויש לדווח עליו ככזה.

כדי להפוך שיפוט זה למוחשי אנו מוסיפים שתי נקודות ייחוס:
- **קו בסיס אקראי** החוזה את המחלקה החיובית בהסתברות השווה לשיעור הבסיס;
- **קו בסיס מבני בן 3 מאפיינים** (דרגה, דרגה משוקללת ומקדם התקבצות), שאינו עושה שימוש באף אחד מ-64 הממדים הנלמדים.

אם ה-embeddings היו נושאים מידע על קריטיות מבנית, היה עליהם לגבור על שניהם בנוחות. הסיבה לצפות אחרת: מסלולי Node2Vec מקודדים *היכן ממוקם צומת בתוך מבנה הקהילות* - באיזו שכונה, באיזה ציר - ולא האם הצומת הוא צומת חיתוך. שתי תחנות באותו רחוב מקבלות וקטורים כמעט זהים גם כאשר אחת מהן היא הגשר היחיד אל פרבר שלם.

In [ ]:
clf_ids = [n for n in Gc.nodes() if n in emb_full]
X_emb = np.vstack([emb_full[n] for n in clf_ids])
y = np.array([critical[n] for n in clf_ids])

clustering = nx.clustering(Gc)
X_struct = np.array([[Gc.degree(n),
                      Gc.degree(n, weight="weight"),
                      clustering.get(n, 0.0)] for n in clf_ids], dtype=float)

idx_tr, idx_te = train_test_split(np.arange(len(clf_ids)), test_size=0.2,
                                  random_state=SEED, stratify=y)

def run_classifier(name, X, clf):
    scaler = StandardScaler().fit(X[idx_tr])
    clf.fit(scaler.transform(X[idx_tr]), y[idx_tr])
    pred = clf.predict(scaler.transform(X[idx_te]))
    f1 = f1_score(y[idx_te], pred, average="binary", zero_division=0)
    print(f"--- {name}: F1 = {f1:.4f}")
    print(classification_report(y[idx_te], pred, target_names=["regular", "critical"],
                                zero_division=0))
    return {"model": name, "f1_critical": round(float(f1), 4),
            "confusion": confusion_matrix(y[idx_te], pred).tolist()}

clf_rows = [
    run_classifier("Embeddings + LogisticRegression", X_emb,
                   LogisticRegression(class_weight="balanced", max_iter=500,
                                      random_state=SEED)),
    run_classifier("Embeddings + RandomForest", X_emb,
                   RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                          random_state=SEED, n_jobs=-1)),
    run_classifier("Degree/clustering + RandomForest", X_struct,
                   RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                          random_state=SEED, n_jobs=-1)),
]

base_rate = float(y.mean())
rand_f1 = 2 * base_rate / (1 + base_rate)  # F1 of predicting positive at the base rate
clf_rows.append({"model": "Random guess at the base rate",
                 "f1_critical": round(rand_f1, 4), "confusion": None})
print(f"Base rate of critical stations: {base_rate:.3f} "
      f"-> a random guesser scores F1 = {rand_f1:.4f}")

clf_df = pd.DataFrame([{k: v for k, v in r.items() if k != "confusion"} for r in clf_rows])
clf_df.to_csv(TABLES / "critical_classifier_results.csv", index=False, encoding="utf-8-sig")
display(clf_df)

cms = [r for r in clf_rows if r["confusion"] is not None]
fig, axes = plt.subplots(1, len(cms), figsize=(5 * len(cms), 4.2))
for ax, r in zip(np.atleast_1d(axes), cms):
    sns.heatmap(np.array(r["confusion"]), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["regular", "critical"], yticklabels=["regular", "critical"],
                ax=ax)
    ax.set_title(f"{r['model']}\nF1 = {r['f1_critical']:.3f}", fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(FIGURES / "confusion_matrices.png", dpi=150)
plt.show()

## מה מרחב ה-embedding באמת לוכד

כישלון הסיווג מעלה את השאלה המתבקשת: אם הווקטורים אינם מקודדים קריטיות, מה הם כן מקודדים? אנו מטילים אותם לשני ממדים וצובעים את הנקודות בשלוש דרכים - לפי אזור גיאוגרפי, לפי קהילת Louvain, ולפי קריטיות. t-SNE משמר שכנויות מקומיות והוא ריבועי במספר הנקודות, ולכן אנו מטילים תת-מדגם אקראי של `TSNE_SAMPLE` צמתים; PCA הוא לינארי ומורץ על הקבוצה המלאה.

הציפייה, שהתרשימים אמורים לאשש, היא שהאזור והקהילה ייפרדו בבירור (מסלולים נותרים בתוך אזור מטרופוליני) ואילו הקריטיות תתפזר באופן אחיד - אותה מסקנה שאליה הגיעו ציוני ה-F1, בתצוגה ישירה.

In [ ]:
t0 = time.time()
communities = nx.community.louvain_communities(Gc, seed=SEED, weight="weight")
comm_of = {n: i for i, c in enumerate(sorted(communities, key=len, reverse=True))
           for n in c}
print(f"Louvain: {len(communities):,} communities in {time.time() - t0:.0f}s")

vis_rng = random.Random(SEED)
vis_ids = vis_rng.sample(clf_ids, min(TSNE_SAMPLE, len(clf_ids)))
X_vis = np.vstack([emb_full[n] for n in vis_ids])

t0 = time.time()
perp = min(30, max(5, len(vis_ids) - 1))
try:
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=perp, max_iter=300, init="pca")
except TypeError:  # scikit-learn < 1.5 calls the parameter n_iter
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=perp, n_iter=300, init="pca")
XY = tsne.fit_transform(X_vis)
print(f"t-SNE on {len(vis_ids):,} nodes in {time.time() - t0:.0f}s")

def scatter_by(labels, title, fname, max_cats=12):
    counts = pd.Series(labels).value_counts()
    keep = list(counts.index[:max_cats])
    cmap = matplotlib.colormaps.get_cmap("tab20").resampled(max(len(keep), 2))
    fig, ax = plt.subplots(figsize=(9.5, 7.5))
    other = [i for i, l in enumerate(labels) if l not in keep]
    if other:
        ax.scatter(XY[other, 0], XY[other, 1], s=4, alpha=0.25, color="#d1d5db",
                   label="other")
    for ci, lab in enumerate(keep):
        sel = [i for i, l in enumerate(labels) if l == lab]
        ax.scatter(XY[sel, 0], XY[sel, 1], s=5, alpha=0.6, color=cmap(ci), label=str(lab))
    ax.legend(markerscale=3, fontsize=8, loc="best")
    ax.set_title(title)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    plt.tight_layout()
    plt.savefig(FIGURES / fname, dpi=150)
    plt.show()

scatter_by([Gc.nodes[n].get("region", "?") or "?" for n in vis_ids],
           "t-SNE of station embeddings, coloured by geographic region",
           "tsne_by_region.png")
scatter_by([f"community {comm_of.get(n, -1)}" for n in vis_ids],
           "t-SNE of station embeddings, coloured by Louvain community (12 largest)",
           "tsne_by_community.png")
scatter_by(["critical" if critical[n] else "regular" for n in vis_ids],
           "t-SNE of station embeddings, coloured by criticality (no visible structure)",
           "tsne_by_criticality.png", max_cats=2)

pca = PCA(n_components=2, random_state=SEED)
P = pca.fit_transform(X_emb)
fig, ax = plt.subplots(figsize=(9, 7))
cols = ["#dc2626" if critical[n] else "#94a3b8" for n in clf_ids]
ax.scatter(P[:, 0], P[:, 1], s=4, alpha=0.4, c=cols)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#94a3b8", label="regular"),
                   Patch(color="#dc2626", label="critical")])
ax.set_title(f"PCA of station embeddings "
             f"({pca.explained_variance_ratio_.sum() * 100:.1f}% of variance explained)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
plt.tight_layout()
plt.savefig(FIGURES / "pca_embeddings.png", dpi=150)
plt.show()

## דמיון בין תחנות: בדיקת שפיות איכותנית

לסיום, בדיקה קריאה לכך שה-embeddings אינם רעש: עבור חמש תחנות לדוגמה אנו מציגים את עשרת שכניהן הקרובים ביותר במרחב ה-cosine, תוך שימוש חוזר באינדקס השכנים הקרובים שנבנה עבור הצעת הקישורים. אם הווקטורים בעלי משמעות, שכניה של תחנה צריכים להיות תחנות על אותו ציר או באותה עיר. זהו בדיוק מידע השכנות המקומית שתרשימי ה-t-SNE מראים ושמסווג הקריטיות לא הצליח לנצל.

In [ ]:
sample_positions = [0, len(emb_ids) // 5, len(emb_ids) // 3,
                    len(emb_ids) // 2, len(emb_ids) * 3 // 4]
sim_rows = []
for pos in sample_positions:
    target = emb_ids[pos]
    for rank, (d, j) in enumerate(zip(nn_dist[pos][1:11], nn_idx[pos][1:11]), 1):
        other = emb_ids[j]
        sim_rows.append({
            "target_stop": target,
            "target_name": attr_of(target, "stop_name"),
            "rank": rank,
            "similar_stop": other,
            "similar_name": attr_of(other, "stop_name"),
            "same_region": attr_of(target, "region") == attr_of(other, "region"),
            "is_existing_edge": Gc.has_edge(target, other),
            "cosine_similarity": round(1.0 - float(d), 4),
        })
sim_df = pd.DataFrame(sim_rows)
sim_df.to_csv(TABLES / "station_similarity.csv", index=False, encoding="utf-8-sig")
print(f"Share of nearest neighbours in the same region: "
      f"{sim_df['same_region'].mean() * 100:.1f}%")
display(sim_df[sim_df["rank"] <= 3][["target_name", "rank", "similar_name",
                                     "same_region", "cosine_similarity"]])

## מסקנות

1. **תוצאת ה-Node2Vec המרכזית הייתה ארטיפקט של דליפה.** אימון ה-embeddings על הגרף המלא ולאחר מכן בדיקה על קשתות שהיו נוכחות במהלך אותו אימון הניבו AUC 0.9928. עם אותו מודל בדיוק, מאומן על גרף האימון בלבד, ה-AUC יורד לערך המודפס בטבלת התוצאות לעיל. הפער הוא גודלה של השפעת השינון, ולא שיפור מודלי. קו הבסיס הקלאסי הדולף במכוון (שכנים משותפים המחושבים על הגרף המלא) מתנהג באותו אופן, מה שמאשש שהסיבה היא הפרוטוקול ולא דבר מה הייחודי ל-embeddings.

2. **סדר הפעולות הוא כל הלקח.** תחילה לפצל, ורק אז ללמוד. כל ייצוג הנלמד על גרף שעדיין מכיל את קשתות המבחן - embeddings, GNNs, פירוק מטריצות - כבר ראה את התשובה.

3. **רוב ה-AUC שנותר הוא תכונה של המדגם השלילי.** מול זוגות שאינם קשת שנבחרו אקראית, כל שיטה נראית מכובדת; מול זוגות שאינם קשת במרחק 2 צעדים (זוגות שיכלו באמת להיות מחוברים) כל הציונים צונחים. כל ערך AUC המצוטט עבור link prediction חסר משמעות ללא ציון האופן שבו נדגמו הזוגות השליליים.

4. **המדד שתויג "Common Neighbors" בתוצאות המקוריות היה CCPA.** `nx.common_neighbor_centrality` עם ברירת המחדל `alpha = 0.8` מוסיף איבר של מרחק מסלול קצר ביותר גלובלי, וזו הסיבה שהוא קיבל 0.8337 בעוד ש-Jaccard, Adamic-Adar ו-Resource Allocation - מדדי חפיפה מקומית אמיתיים - עמדו כולם על 0.6301. עצם ההסכמה של שלושת המדדים המקומיים עד ארבע ספרות אחרי הנקודה היא כשלעצמה אות: בגרף שדרגתו הממוצעת 3.4, רוב הזוגות חולקים אפס שכנים או שכן אחד, ולכן שלוש הנוסחאות מדרגות אותם כמעט באופן זהה.

5. **הוספת הקישורים המוצעים אינה משפרת את החוסן.** המקור דיווח 0.9965 -> 0.9965 (0.0%+) תחת כותרת המבטיחה שיפור. סריקת תקציב ההסרה מ-50 ועד 2,000 תחנות אינה מצילה את התוצאה. עשרים קשתות מתוך 52,000, הממוקמות בין תחנות הדומות מבחינה מבנית (ולכן בדרך כלל כבר מקושרות היטב לשכנויות זו של זו), אינן יכולות להזיז מדד קישוריות גלובלי. שיפור חוסנה של רשת זו מחייב קישורים הנבחרים כדי לגשר על צמתי חיתוך, ולא קישורים הנבחרים לפי דמיון embedding.

6. **רשימת הקישורים המוצעים היא כעת דירוג גלובלי אמיתי.** המקור דירג רק זוגות בתוך מדגם אקראי של 500 תחנות - כ-0.03% מן הזוגות האפשריים - וכינה את התוצאה עשרים המובילים של הרשת. חיפוש ה-cosine המדויק על כל הזוגות המשמש כאן משנה את הרשימה כליל, ועמודת המרחק מראה כמה מזוגות הדמיון המובילים אינם סבירים גיאוגרפית כקטעים חדשים.

7. **node embeddings אינם מזהים תחנות קריטיות.** ה-F1 נותר קרוב לרמתו של מנחש לפי שיעור הבסיס ואינו גובר על קו בסיס בן שלושה מאפיינים של דרגה והתקבצות. זוהי תוצאה שלילית והיא תוצאה קוהרנטית: מסלולים אקראיים מקודדים השתייכות לקהילה וגיאוגרפיה, כפי שתרשימי ה-t-SNE מראים בבירור, ואילו מעמד של נקודת חיתוך הוא תכונת חתך גלובלית שאין בכוחן של סטטיסטיקות מופע-משותף מקומיות לחשוף. לאיתור תחנות קריטיות, האלגוריתמים המבניים הישירים שבהם נעשה שימוש קודם לכן בפרויקט זה - articulation points ו-betweenness - נותרים הכלי הנכון.